## LIBRERIA

In [4]:
from sqlalchemy import create_engine
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import mapclassify 
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

## 1.CONEXIÓN BASE DE DATOS

In [5]:
DB_HOST = 'localhost'
DB_PORT = 5432
DB_NAME = 'CensoRM2017'
DB_USER = 'postgres'
DB_PASSWORD = 'postgres' 

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

## 2.CARGAR DATOS

In [6]:
sql_comunas = '''
SELECT nom_comuna, geom
FROM dpa.comunas_rm_shp;
'''

gdf_comunas    = gpd.read_postgis(sql_comunas, engine, geom_col='geom')
gdf_centroides = gdf_comunas.copy()
gdf_centroides['geometry'] = gdf_comunas.centroid

print(f'Comunas: {len(gdf_comunas)}')
display(gdf_comunas)

Comunas: 52


,nom_comuna,geom
0,LO PRADO,"MULTIPOLYGON (((339891.406 6298725.5, 339967.9..."
1,INDEPENDENCIA,"MULTIPOLYGON (((345568.713 6303028.6, 345794.7..."
2,LO ESPEJO,"MULTIPOLYGON (((344800.111 6290835.676, 343521..."
3,SAN RAMÓN,"MULTIPOLYGON (((348001.245 6290000.561, 348259..."
4,LA CISTERNA,"MULTIPOLYGON (((346490 6286649.498, 345256.375..."
5,PEDRO AGUIRRE CERDA,"MULTIPOLYGON (((344800.111 6290835.676, 342530..."
6,SAN MIGUEL,"MULTIPOLYGON (((348011.801 6290001.611, 348001..."
7,CONCHALÍ,"MULTIPOLYGON (((342803.041 6306876.533, 343470..."
8,SAN JOAQUÍN,"MULTIPOLYGON (((349191.057 6294882.314, 349437..."
9,LA GRANJA,"MULTIPOLYGON (((350573.769 6287953.638, 350492..."


## 3.INDICADORES

In [7]:
sql_indicadores = '''
WITH agg AS 
(
SELECT c.nom_comuna, 
z.geocodigo::DOUBLE PRECISION AS geocodigo, 
ROUND (((COUNT(*) FILTER (WHERE UPPER(p.p18) IN ('A', 'B')))*100.0/COUNT(*)),2) AS sector_primario,
ROUND (((COUNT(*) FILTER (WHERE p.p15 = 12))*100.0/COUNT(*)),2) AS porcentaje_profesionales
FROM public.personas AS p
JOIN public.hogares AS h
ON p.hogar_ref_id = h.hogar_ref_id
JOIN public.viviendas AS v
ON h.vivienda_ref_id = v.vivienda_ref_id
JOIN public.zonas AS z
ON v.zonaloc_ref_id = z.zonaloc_ref_id
JOIN public.comunas AS c
ON z.codigo_comuna = c.codigo_comuna
JOIN public.provincias AS pr 
ON pr.provincia_ref_id = c.provincia_ref_id
WHERE pr.nom_provincia = 'MELIPILLA'
GROUP BY c.nom_comuna, z.geocodigo
)
SELECT a.*, shp.geom
FROM agg AS a
JOIN dpa.zonas_censales_rm AS shp
ON shp.geocodigo = a.geocodigo;
'''

In [8]:
gdf = gpd.read_postgis(sql_indicadores, engine, geom_col='geom')

In [9]:
gdf

,nom_comuna,geocodigo,sector_primario,porcentaje_profesionales,geom
0,ALHUÉ,1.350201e+10,18.37,9.35,"MULTIPOLYGON (((306250.709 6232826.305, 306344..."
1,ALHUÉ,1.350201e+10,14.58,0.00,"MULTIPOLYGON (((304044.672 6232382.536, 304070..."
2,ALHUÉ,1.350202e+10,34.98,3.67,"MULTIPOLYGON (((295472.326 6234041.035, 295780..."
3,ALHUÉ,1.350202e+10,45.07,14.08,"MULTIPOLYGON (((291374.242 6233598.206, 291459..."
4,ALHUÉ,1.350203e+10,16.65,5.22,"MULTIPOLYGON (((311500.554 6233729.707, 311534..."
...,...,...,...,...,...
219,SAN PEDRO,1.350506e+10,19.27,6.42,"MULTIPOLYGON (((270726.55 6239669.063, 270758...."
220,SAN PEDRO,1.350506e+10,17.96,5.83,"MULTIPOLYGON (((263857.95 6238016.555, 264500...."
221,SAN PEDRO,1.350506e+10,19.80,2.48,"MULTIPOLYGON (((266072.646 6242106.632, 266163..."
222,SAN PEDRO,1.350506e+10,20.69,1.38,"MULTIPOLYGON (((263628.433 6239389.385, 263669..."


In [10]:
gdf[['sector_primario','porcentaje_profesionales']].describe().round(2)

,sector_primario,porcentaje_profesionales
count,224.00,224.00
mean,13.69,9.19
std,9.49,7.50
min,0.00,0.00
25%,6.49,4.40
50%,12.50,7.09
75%,18.39,11.34
max,75.00,41.10
